# 07 - Task 3: Soft-vote ensemble of the linear family

Combines the three tuned linear models (Ridge, ElasticNet, LinearSVC) by averaging
predicted probabilities, evaluated under the same locked CV and holdout protocol.

**Why the linear family specifically:** The Kaggle results invert the local ranking:
LightGBM wins local CV by ~0.015 but loses the leaderboard by ~0.029. The linear models
transport across the documented train-to-test shift noticeably better, so the ensemble
is built from them rather than from the best local scorers.

 Ridge and ElasticNet are both logistic
regression differing only in penalty, so their predictions will be heavily correlated
and averaging them alone should buy very little. Most of any real gain has to come from
LinearSVC's different objective (max-margin hinge rather than log-loss). Section 3
measures pairwise disagreement explicitly so the ensemble's ceiling is visible rather
than assumed - if members agree on nearly every row, there is nothing to gain and this
notebook should conclude that rather than shipping a submission anyway.

## 0. Setup

In [ ]:
# Adds project root to path so `import src...` works from notebooks/.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import json
import joblib
import itertools

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB

from src import paths, data, evaluation
from src.paths import FIGURES
FIGURES.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'src'

## 1. Load features + the locked split

In [ ]:
X, y, ids = data.load_train_features()
Xt, test_ids = data.load_test_features()
dev_idx = np.load(paths.DATA_PROCESSED / 'dev_idx.npy')
holdout_idx = np.load(paths.DATA_PROCESSED / 'holdout_idx.npy')
cv = evaluation.make_cv()

assert len(dev_idx) == 16000 and len(holdout_idx) == 4000
assert not set(dev_idx) & set(holdout_idx), "dev and holdout overlap"

X_dev, y_dev = X[dev_idx], y[dev_idx]
X_hold, y_hold = X[holdout_idx], y[holdout_idx]
print(f"dev {X_dev.shape}, holdout {X_hold.shape}")

## 2. Rebuild the tuned members from their persisted params

Each member is reconstructed from the `best_*_params.json` written by its own tuning
notebook, so this notebook never re-searches and never drifts from what was tuned.

LinearSVC has no `predict_proba` (it optimizes margins, not likelihoods), so it is
wrapped in `CalibratedClassifierCV(method="sigmoid")` - Platt scaling, which fits a
one-dimensional logistic map from margin to probability using internal cross-validation.
This is what makes it usable in a soft vote at all.

`ComplementNB` is included as a **candidate** fourth member, not a committed one. It is
much weaker alone (04_models baseline 0.6580) but has a genuinely different inductive
bias - a generative model rather than a discriminative one - so it may contribute
diversity that the three near-identical linear discriminators cannot. Section 4 decides
whether to keep it based on measured CV.

In [ ]:
def load_params(name):
    with open(paths.DATA_PROCESSED / f"best_{name}_params.json") as f:
        return json.load(f)


ridge_p = load_params("logreg_ridge")
enet_p = load_params("logreg_elasticnet")
svc_p = load_params("linearsvc")
print("ridge     :", ridge_p)
print("elasticnet:", enet_p)
print("linearsvc :", svc_p)

# class_weight is fixed to "balanced" for every linear member (established in 05b); drop
# it from the loaded dicts so it is not passed twice.
strip = lambda d: {k: v for k, v in d.items() if k != "class_weight"}

members = {
    "logreg_ridge": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42, **strip(ridge_p),
    ),
    "logreg_elasticnet": LogisticRegression(
        penalty="elasticnet", solver="saga", max_iter=2000,
        class_weight="balanced", random_state=42, **strip(enet_p),
    ),
    "linearsvc_cal": CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", max_iter=5000, dual="auto",
                  random_state=42, **strip(svc_p)),
        method="sigmoid", cv=5,
    ),
    "complementnb": ComplementNB(),
}
for name, est in members.items():
    print(f"{name:20s} {type(est).__name__}")

## 3. Out-of-fold probabilities + member diversity

Every member gets out-of-fold probabilities on `dev_idx` under the identical locked
folds, so member scores and the ensemble score are all computed on the same rows and
are directly comparable.

This cell is the slow one: `saga` (required for the elasticnet penalty) runs roughly 4
minutes per fit, and the calibrated LinearSVC fits 5 inner models per outer fold. Estimated Running Time: half an hour.

In [ ]:
oof = {}
for name, est in members.items():
    print(f"OOF: {name} ...", flush=True)
    oof[name] = cross_val_predict(
        clone(est), X_dev, y_dev, cv=cv, method="predict_proba",
    )[:, 1]
    f1 = evaluation.macro_f1(y_dev, (oof[name] >= 0.5).astype(int))
    print(f"  OOF Macro F1: {f1:.4f}")

np.savez(paths.DATA_PROCESSED / "oof_linear_members.npz", **oof)
print("\nSaved oof_linear_members.npz")

In [ ]:
# Diversity check: how often do members actually disagree on the final label? An
# ensemble can only fix rows where members differ, so this bounds the possible gain.
labels = {n: (p >= 0.5).astype(int) for n, p in oof.items()}
names = list(labels)

print("Pairwise disagreement rate (fraction of dev rows with different labels):")
dis = pd.DataFrame(index=names, columns=names, dtype=float)
for a, b in itertools.product(names, names):
    dis.loc[a, b] = float((labels[a] != labels[b]).mean())
print(dis.round(4).to_string())

print("\nProbability correlation (Pearson) between members:")
print(pd.DataFrame(oof).corr().round(4).to_string())

## 4. Choose the member set

Evaluates the equal-weight soft vote over every candidate subset of at least two
members and reports each against the best individual member. The subset that wins on
CV is carried forward - including, if that is what the numbers say, a subset that
excludes ComplementNB or excludes one of the two logistic models.

In [ ]:
def soft_vote_f1(subset, proba_map, y_true):
    """Macro F1 of the equal-weight probability average over `subset`."""
    avg = np.mean([proba_map[n] for n in subset], axis=0)
    return evaluation.macro_f1(y_true, (avg >= 0.5).astype(int))


member_f1 = {n: evaluation.macro_f1(y_dev, labels[n]) for n in names}
best_member = max(member_f1, key=member_f1.get)
print("Individual OOF Macro F1:")
for n, v in sorted(member_f1.items(), key=lambda kv: -kv[1]):
    print(f"  {n:20s} {v:.4f}")
print(f"\nBest single member: {best_member} ({member_f1[best_member]:.4f})\n")

rows = []
for r in range(2, len(names) + 1):
    for subset in itertools.combinations(names, r):
        rows.append({
            "members": " + ".join(subset),
            "n": r,
            "cv_f1": soft_vote_f1(subset, oof, y_dev),
            "subset": subset,
        })
subsets = pd.DataFrame(rows).sort_values("cv_f1", ascending=False).reset_index(drop=True)
subsets["vs_best_member"] = subsets["cv_f1"] - member_f1[best_member]
print(subsets[["members", "n", "cv_f1", "vs_best_member"]].to_string(index=False))

chosen = subsets.iloc[0]
chosen_members = list(chosen["subset"])
ensemble_cv_f1 = float(chosen["cv_f1"])
beats_best_member = ensemble_cv_f1 > member_f1[best_member]

print(f"\nChosen subset: {chosen['members']}")
print(f"Ensemble CV Macro F1: {ensemble_cv_f1:.4f} "
      f"({chosen['vs_best_member']:+.4f} vs best single member)")
print(f"Beats best single member: {beats_best_member}")

## 5. Holdout check

Members are fit on `dev_idx` only and evaluated once on the untouched holdout.

In [ ]:
hold_proba = {}
for name in chosen_members:
    est = clone(members[name])
    est.fit(X_dev, y_dev)
    hold_proba[name] = est.predict_proba(X_hold)[:, 1]
    print(f"{name:20s} holdout F1 "
          f"{evaluation.macro_f1(y_hold, (hold_proba[name] >= 0.5).astype(int)):.4f}")

ens_hold_proba = np.mean([hold_proba[n] for n in chosen_members], axis=0)
ens_hold_pred = (ens_hold_proba >= 0.5).astype(int)
holdout_f1 = evaluation.macro_f1(y_hold, ens_hold_pred)
gap = ensemble_cv_f1 - holdout_f1

print(f"\nEnsemble CV Macro F1:      {ensemble_cv_f1:.4f}")
print(f"Ensemble holdout Macro F1: {holdout_f1:.4f}")
print(f"Gap (CV - holdout): {gap:+.4f}")
print(f"Predicted machine-class share: {ens_hold_pred.mean():.4f} "
      f"(true {y_hold.mean():.4f})")

In [ ]:
metrics_path = paths.DATA_PROCESSED / "holdout_metrics.csv"
new_row = pd.DataFrame([{
    "model": "linear_ensemble",
    "cv_mean_f1": ensemble_cv_f1,
    "holdout_f1": holdout_f1,
    "gap": gap,
}])
if metrics_path.exists():
    existing = pd.read_csv(metrics_path)
    existing = existing[existing["model"] != "linear_ensemble"]
    holdout_metrics = pd.concat([existing, new_row], ignore_index=True)
else:
    holdout_metrics = new_row
holdout_metrics.to_csv(metrics_path, index=False)
print(holdout_metrics.to_string(index=False))

## 6. Refit on full train + write submission

Guarded: the submission is only written if the ensemble actually beat its best single
member on CV. If it did not, the honest conclusion is that the members are too
correlated to combine usefully, and shipping it anyway would just be noise.

In [ ]:
if not beats_best_member:
    print("Ensemble did NOT beat its best single member on CV "
          f"({ensemble_cv_f1:.4f} vs {member_f1[best_member]:.4f}).")
    print("No submission written - see the discussion section.")
else:
    fitted = {}
    for name in chosen_members:
        est = clone(members[name])
        est.fit(X, y)  # all labeled data
        fitted[name] = est
    joblib.dump(fitted, paths.MODELS / "final_linear_ensemble_full_refit.pkl")

    test_proba = np.mean([fitted[n].predict_proba(Xt)[:, 1] for n in chosen_members], axis=0)
    preds = (test_proba >= 0.5).astype(int)

    # Assert alignment BEFORE writing - a misaligned id column silently destroys the
    # submission and the leaderboard gives no hint that it happened.
    sample = pd.read_csv(paths.DATA_RAW / "sample_submission.csv", dtype={"id": str})
    assert len(test_ids) == 6999, len(test_ids)
    assert list(test_ids) == list(sample["id"]), "test ids do not match sample_submission order"

    data.write_submission(test_ids, preds, "linear_ensemble_task3.csv")

    with open(paths.DATA_PROCESSED / "best_linear_ensemble_params.json", "w") as f:
        json.dump({
            "members": chosen_members,
            "weights": "equal",
            "combine": "mean predict_proba, threshold 0.5",
            "cv_mean_f1": ensemble_cv_f1,
            "holdout_f1": holdout_f1,
            "member_cv_f1": member_f1,
        }, f, indent=2)

    bal = pd.Series(preds).value_counts(normalize=True).round(4).to_dict()
    print(f"Wrote linear_ensemble_task3.csv - members {chosen_members}")
    print(f"  CV {ensemble_cv_f1:.4f}, holdout {holdout_f1:.4f}")
    print(f"  predicted balance {bal} (train {pd.Series(y).value_counts(normalize=True).round(4).to_dict()})")

## Discussion / carry-forward -> Task 4 report

**Chosen ensemble: ElasticNet + calibrated LinearSVC, equal weights. CV 0.7307
(+0.0017 over the best single member), holdout 0.7396.**

**The expectation set in the header was correct:** Ridge and ElasticNet have a probability correlation of 0.984
and disagree on only 5.5% of dev rows, and combining just those two was actually
*negative* (-0.0008 versus ElasticNet alone). Every subset that gained did so by
including LinearSVC, which disagrees with the logistic models on 11-12% of rows. Diversity,
not member quality, is what the soft vote was able to exploit.

**ComplementNB was tested and rejected on the evidence.** It is by far the most diverse
member (22-28% disagreement, correlation 0.79-0.84) but at 0.6579 CV it is too weak, and
it degraded every subset it joined. Diversity is necessary but not sufficient.

**The gain is small and honest: +0.0017 CV.** That is well inside the CV standard
deviation of roughly 0.004, so on local evidence alone this ensemble is not
distinguishable from ElasticNet by itself.

**Do not submit this file at the default cutoff.** The ensemble predicts **72.17% machine**
on the test set, against ElasticNet's 66.08% and train's 62.52%. Calibrating LinearSVC's
margins into probabilities pushes its test-side scores upward, and averaging drags the
combined share up with it. Every scored submission in that share range landed between
0.665 and 0.676, well below ElasticNet's 0.69576. So the class-balance evidence predicts
this ensemble would *lose* around 0.02 Macro F1 on Kaggle while winning 0.0017 on CV, and
it is excluded from today's upload batch for that reason.

This is the sharpest illustration in the project of why local validation is not a safe
guide here: a model that is locally better and is built entirely from the two
best-transporting members would still have cost a submission slot and a worse score. If
the ensemble is revisited, it should be thresholded upward to bring its predicted share
into the 0.60-0.66 band first, using whatever notebook 08 establishes as the optimum.

**Carry forward:** `submissions/linear_ensemble_task3.csv` exists but is **held back**.
Notebook 08 owns the four submissions being uploaded today.